In [16]:
import json
import re
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO

import pandas as pd
import requests

In [17]:
FAMILY_MAP = {
    "Aire acondicionado": "AA",
    "Estacion de Carga Portatil": "GENKI",
    "TPMS": "TPMS",
    "TPMS Repuesto": "TPMS",
    "Climatizador": "CLIMATIZADOR",
    "Carjack": "CARJACK",
    "Arrancador": "CARJACK",
    "Inflador": "CARJACK",
    "Caldera": "CALDERA",
    "Solar": "SOLAR",
}

In [18]:
def normalize_sku(x) -> str:
    return str(x).strip()


def split_images(s: str) -> list[str]:
    if not isinstance(s, str) or not s.strip():
        return []
    return [u.strip() for u in s.split(",") if u.strip()]


def to_bool(x):
    if pd.isna(x):
        return None
    return bool(x)

In [23]:
def export_from_csv(csv_path_or_url: str, out_path: Path):
    if csv_path_or_url.startswith("http"):
        response = requests.get(csv_path_or_url)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text))
    else:
        df = pd.read_csv(csv_path_or_url)

    items = []

    for _, r in df.iterrows():
        sku = normalize_sku(r["id"])
        product_family = r["product_family"] if pd.notna(r["product_family"]) else ""
        model = str(r["model"]).strip() if pd.notna(r["model"]) else ""
        title = str(r["title"]).strip() if pd.notna(r["title"]) else ""
        description = str(r["description"]).strip(
        ) if pd.notna(r["description"]) else ""
        availability = str(r["availability"]).strip().lower() == "in stock"
        condition = str(r["condition"]).strip().lower()
        
        # Parse price: remove " ARS", then handle Argentine format (dots as thousands, comma as decimal)
        if pd.notna(r["price"]):
            price_str = str(r["price"]).replace(" ARS", "").strip()
            price_str = price_str.replace(".", "").replace(",", ".")
            try:
                price_ars = float(price_str)
            except ValueError:
                price_ars = None
        else:
            price_ars = None
        
        if pd.notna(r["sale_price"]):
            sale_price_str = str(r["sale_price"]).replace(" ARS", "").strip()
            sale_price_str = sale_price_str.replace(".", "").replace(",", ".")
            try:
                sale_price_ars = float(sale_price_str)
            except ValueError:
                sale_price_ars = None
        else:
            sale_price_ars = None
        
        link = str(r["link"]).strip() if pd.notna(r["link"]) else None
        image_link = str(r["image_link"]).strip(
        ) if pd.notna(r["image_link"]) else None
        brand = str(r["brand"]).strip() if pd.notna(r["brand"]) else None
        google_product_category = str(r["google_product_category"]).strip(
        ) if pd.notna(r["google_product_category"]) else ""
        custom_label_1 = str(r["custom_label_1"]).strip(
        ) if pd.notna(r["custom_label_1"]) else ""

        # Determine family from google_product_category or title
        fam = "OTHER"
        for key in FAMILY_MAP:
            if key.lower() in product_family.lower() or key.lower() in google_product_category.lower() or key.lower() in title.lower():
                fam = FAMILY_MAP[key]
                break

        items.append({
            "id": sku,
            "model": model,
            "title": title,
            "description": description,
            "availability": "in stock" if availability else "out of stock",
            "condition": condition,
            "price": price_ars,
            "sale_price": sale_price_ars,
            "link": link,
            "image_link": image_link,
            "brand": brand,
            "google_product_category": google_product_category,
            "custom_label_1": custom_label_1,
            "family": fam,
        })

    catalog = {
        "schema_version": "catalog.v1",
        "generated_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "defaults": {"currency": "ARS", "country": "AR"},
        "source": {
            "type": "csv",
            "file": csv_path_or_url,
        },
        "items": sorted(items, key=lambda x: x["id"]),
    }

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(
        catalog, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Wrote {len(catalog['items'])} items to {out_path}")

In [24]:
# Path or URL to CSV source
in_path = "../knowledge/facebook-feed.csv"  # "https://neil.ar/facebook-feed.csv"
out_path = Path("../data/catalog.json")  # Output catalog.json path

if in_path.endswith(".csv") or in_path.startswith("http"):
    export_from_csv(in_path, out_path)
else:
    raise SystemExit("Unsupported input type. Use .csv or URL")

Wrote 247 items to ../data/catalog.json
